In [ ]:
# 1. Imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

# Display / formatting
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

DATA_PATH = Path("../data/processed/mru_usd_ohlc_clean.csv")

In [ ]:
# 2. Load cleaned data
print("Loading cleaned MRU/USD data from:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Basic quick preview
display(df.head(5))
display(df.tail(5))
print("\nColumn dtypes:")
print(df.dtypes)

=== Loading cleaned MRU/USD data ===

=== Head ===
        date      open      high       low     close
0 2000-01-03 22.525000 22.525000 22.525000 22.525000
1 2000-01-04 22.525000 22.525000 22.525000 22.525000
2 2000-01-05 22.525000 22.525000 22.525000 22.525000
3 2000-01-06 22.525000 22.525000 22.525000 22.525000
4 2000-01-07 22.530000 22.530000 22.530000 22.530000

=== Tail ===
           date      open      high       low     close
6747 2025-11-19 39.840000 40.050000 39.640000 39.700000
6748 2025-11-20 39.700000 40.010000 39.581000 39.860000
6749 2025-11-21 39.860000 39.860000 39.581000 39.860000
6750 2025-11-24 39.860000 40.010000 39.592000 39.750000
6751 2025-11-25 39.760000 39.994000 39.663000 39.940000

=== dtypes ===
date     datetime64[ns]
open            float64
high            float64
low             float64
close           float64
dtype: object


In [ ]:
# 3. Basic overview: counts, date range, integrity checks
n_rows = len(df)
n_unique_dates = df["date"].nunique()
date_min = df["date"].min()
date_max = df["date"].max()

print(f"Total rows: {n_rows}")
print(f"Distinct dates: {n_unique_dates}")
print(f"Date range: {date_min.date()} → {date_max.date()}")
print(f"Date sorted ascending? {df['date'].is_monotonic_increasing}")

# Duplicate dates and missing OHLC counts
dup_count = (df["date"].duplicated()).sum()
print(f"Duplicate-date rows: {int(dup_count)}")

ohlc_cols = ["open", "high", "low", "close"]
print("\nMissing values per OHLC column:")
print(df[ohlc_cols].isna().sum())

print("\nOHLC descriptive statistics:")
display(df[ohlc_cols].describe().T)


=== BASIC OVERVIEW ===
Total rows in cleaned series: 6752
Distinct dates: 6752
Date range: 2000-01-03 → 2025-11-25
Number of dates with duplicates (should be 0): 0
Is date column sorted ascending? True

Missing values per OHLC column:
open     0
high     0
low      0
close    0
dtype: int64

OHLC descriptive statistics:
             open        high         low       close
count 6752.000000 6752.000000 6752.000000 6752.000000
mean    30.952400   31.067677   30.895595   30.994285
std      5.236764    5.274372    5.157133    5.212993
min     20.170000   20.170000   20.170000   20.170000
25%     26.360500   26.446250   26.445000   26.505000
50%     29.250000   29.272000   29.018000   29.250000
75%     36.210000   36.300000   36.130000   36.210000
max     40.301000   41.210000   40.070000   40.300000


In [ ]:
# 4. Weekday distribution
df["weekday"] = df["date"].dt.weekday  # 0=Mon, 6=Sun
weekday_counts = df["weekday"].value_counts().sort_index()

weekday_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
weekday_summary = pd.DataFrame({
    "weekday": weekday_counts.index,
    "weekday_name": [weekday_map[w] for w in weekday_counts.index],
    "count": weekday_counts.values
})

display(weekday_summary)
has_weekend = bool(weekday_counts.get(5, 0) + weekday_counts.get(6, 0) > 0)
print(f"Weekend rows present (Sat/Sun)? {has_weekend}")


=== CALENDAR PATTERN (WEEKDAY DISTRIBUTION) ===
   weekday weekday_name  count
0        0          Mon   1351
1        1          Tue   1351
2        2          Wed   1349
3        3          Thu   1350
4        4          Fri   1351

Any weekend (Sat/Sun) rows present in cleaned series? False


In [ ]:
# 5. Price-level summary and period breakdown
print("Close price summary:")
display(df["close"].describe())

# Rough period definitions (adjust dates to your data if needed)
price_periods = [
    ("2000-01-01", "2009-12-31", "2000–2009"),
    ("2010-01-01", "2014-12-31", "2010–2014"),
    ("2015-01-01", "2019-12-31", "2015–2019"),
    ("2020-01-01", date_max.strftime("%Y-%m-%d"), "2020–end"),
]

rows = []
for start, end, label in price_periods:
    mask = (df["date"] >= pd.to_datetime(start)) & (df["date"] <= pd.to_datetime(end))
    sub = df.loc[mask, "close"]
    if sub.empty:
        continue
    rows.append({
        "period": label,
        "start": start,
        "end": end,
        "count": int(len(sub)),
        "min_close": float(sub.min()),
        "max_close": float(sub.max()),
        "mean_close": float(sub.mean()),
    })

if rows:
    price_periods_df = pd.DataFrame(rows)
    display(price_periods_df)

# Plot close price (interactive)
fig_close = px.line(df, x="date", y="close", title="MRU/USD Close Price (cleaned)")
fig_close.show()


=== PRICE LEVEL SUMMARY ===
Close price descriptive statistics:
count   6752.000000
mean      30.994285
std        5.212993
min       20.170000
25%       26.505000
50%       29.250000
75%       36.210000
max       40.300000
Name: close, dtype: float64



Close price summary by rough periods:
                period       start         end  count  min_close  max_close  mean_close
0  Period 1: 2000–2009  2000-01-01  2009-12-31   2604  20.170000  28.550000   25.939843
1  Period 2: 2010–2014  2010-01-01  2014-12-31   1304  26.100000  30.850000   28.845110
2  Period 3: 2015–2019  2015-01-01  2019-12-31   1304  28.400000  37.655000   35.093423
3   Period 4: 2020–end  2020-01-01  2025-11-25   1540  34.220000  40.300000   37.889748


In [ ]:
# 6. Compute returns
print("Computing log and simple returns")

zero_close_count = int((df["close"] == 0).sum())
print(f"Rows with close == 0: {zero_close_count}")

df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["simple_return"] = df["close"].pct_change()

ret = df["log_return"].dropna()
print("\nLog return summary:")
display(ret.describe())

quantiles = ret.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nSelected log-return quantiles:")
display(quantiles)

# Plot log returns
fig_ret = px.line(df, x="date", y="log_return", title="Daily Log Returns")
fig_ret.show()


=== RETURNS COMPUTATION ===
Number of rows with close == 0: 0

Log return descriptive statistics:
count   6751.000000
mean       0.000085
std        0.010057
min       -0.219999
25%        0.000000
50%        0.000000
75%        0.000000
max        0.221456
Name: log_return, dtype: float64

Simple return descriptive statistics:
count   6751.000000
mean       0.000136
std        0.010132
min       -0.197480
25%        0.000000
50%        0.000000
75%        0.000000
max        0.247893
Name: simple_return, dtype: float64

Log return quantiles:
0.010000   -0.022529
0.050000   -0.006418
0.250000    0.000000
0.500000    0.000000
0.750000    0.000000
0.950000    0.006307
0.990000    0.023060
Name: log_return, dtype: float64


In [ ]:
# 7. Returns by year and era
yearly_stats = (
    df.dropna(subset=["log_return"])
      .groupby(df["date"].dt.year)["log_return"]
      .agg(["count", "mean", "std", "min", "max"])
      .reset_index()
      .rename(columns={"date": "year"})
)

display(yearly_stats.head(20))

# assign simple eras for structural overview
def assign_era(y):
    if y <= 2004:
        return "2000–2004"
    if y <= 2009:
        return "2005–2009"
    if y <= 2014:
        return "2010–2014"
    if y <= 2019:
        return "2015–2019"
    return "2020–"

df["era"] = df["date"].dt.year.apply(assign_era)

era_stats = (
    df.dropna(subset=["log_return"])
      .groupby("era")["log_return"]
      .agg(["count", "mean", "std", "min", "max"])
      .reset_index()
)
display(era_stats)

fig_era_vol = px.bar(era_stats, x="era", y="std", title="Std(log return) by era")
fig_era_vol.show()


=== RETURNS BY YEAR ===
    year  count      mean      std       min      max
0   2000    259  0.000469 0.003898 -0.007797 0.044704
1   2001    260  0.000217 0.033124 -0.219999 0.221456
2   2002    260 -0.000107 0.004099 -0.022494 0.022393
3   2003    259 -0.000071 0.006821 -0.060859 0.061009
4   2004    261  0.000069 0.016498 -0.116731 0.116928
5   2005    260  0.000295 0.007555 -0.051017 0.051017
6   2006    260 -0.000160 0.003821 -0.045257 0.025083
7   2007    261 -0.000301 0.009140 -0.059487 0.060600
8   2008    262  0.000143 0.009948 -0.038505 0.042840
9   2009    261  0.000043 0.005288 -0.023392 0.023392
10  2010    261  0.000281 0.003019 -0.017062 0.017062
11  2011    260  0.000081 0.004648 -0.037271 0.008569
12  2012    261  0.000162 0.002660 -0.015013 0.015013
13  2013    261 -0.000129 0.013012 -0.042487 0.128061
14  2014    261  0.000020 0.008147 -0.065295 0.053257
15  2015    261  0.000241 0.021193 -0.087706 0.111602
16  2016    261  0.000527 0.008052 -0.034387 0.075451
17 

In [ ]:
# 8. Rolling volatility
window_short = 30
window_long = 90

df["vol_30"] = df["log_return"].rolling(window_short).std()
df["vol_90"] = df["log_return"].rolling(window_long).std()

print("30-day vol (non-NaN) stats:")
display(df["vol_30"].dropna().describe())

print("90-day vol (non-NaN) stats:")
display(df["vol_90"].dropna().describe())

# Save rolling vol (derived)
derived_dir = Path("../data/derived")
derived_dir.mkdir(parents=True, exist_ok=True)
vol_path = derived_dir / "mru_usd_rolling_volatility.csv"
df.loc[:, ["date", "vol_30", "vol_90"]].to_csv(vol_path, index=False)
print("Saved rolling volatility to:", vol_path)

# Plot rolling volatility (interactive)
fig_vol = px.line(df, x="date", y=["vol_30", "vol_90"], title="Rolling volatility (30d, 90d)")
fig_vol.show()

# Top volatility dates
N_top = 10
top_vol30 = df[["date", "vol_30"]].dropna().nlargest(N_top, "vol_30")
top_vol90 = df[["date", "vol_90"]].dropna().nlargest(N_top, "vol_90")

print(f"Top {N_top} dates by 30-day volatility:")
display(top_vol30.reset_index(drop=True))
print(f"Top {N_top} dates by 90-day volatility:")
display(top_vol90.reset_index(drop=True))


=== ROLLING VOLATILITY ===

30-day rolling vol stats (on non-NaN values):
count   6722.000000
mean       0.005705
std        0.008472
min        0.000000
25%        0.001749
50%        0.003166
75%        0.006356
max        0.080857
Name: vol_30, dtype: float64

90-day rolling vol stats (on non-NaN values):
count   6662.000000
mean       0.006559
std        0.007769
min        0.000200
25%        0.002508
50%        0.003912
75%        0.007650
max        0.056159
Name: vol_90, dtype: float64

Rolling volatility saved to: data\derived\mru_usd_rolling_volatility.csv



Top 10 dates by 30-day rolling volatility:
          date   vol_30
315 2001-03-20 0.080857
316 2001-03-21 0.080857
317 2001-03-22 0.080857
318 2001-03-23 0.080857
313 2001-03-16 0.080857
314 2001-03-19 0.080857
312 2001-03-15 0.080832
336 2001-04-18 0.079524
332 2001-04-12 0.079524
333 2001-04-13 0.079524

Top 10 dates by 90-day rolling volatility:
          date   vol_90
367 2001-05-31 0.056159
368 2001-06-01 0.056159
369 2001-06-04 0.056159
370 2001-06-05 0.056159
377 2001-06-14 0.056155
378 2001-06-15 0.056155
365 2001-05-29 0.056149
366 2001-05-30 0.056149
372 2001-06-07 0.056147
373 2001-06-08 0.056147


In [ ]:
# 9. Helpers for choosing modeling period
print(f"Series date range: {date_min.date()} → {date_max.date()}")
print(f"Total observations: {n_rows}")

# observations per year
obs_per_year = df.groupby(df["date"].dt.year).size().reset_index(name="count").rename(columns={"date":"year"})
display(obs_per_year.head(12))

# observations per era
obs_per_era = df.groupby("era").size().reset_index(name="count")
display(obs_per_era)

# observations per month (sample)
df["year_month"] = df["date"].dt.to_period("M")
obs_per_month = df.groupby("year_month").size().reset_index(name="count")
display(obs_per_month.head(24))


=== HELPERS FOR MODELING PERIOD CHOICE ===
Cleaned series date range: 2000-01-03 → 2025-11-25
Total observations: 6752

Number of observations per year:
    year  count
0   2000    260
1   2001    260
2   2002    260
3   2003    259
4   2004    261
..   ...    ...
21  2021    261
22  2022    260
23  2023    260
24  2024    262
25  2025    235

[26 rows x 2 columns]

Number of observations per era:
                era  count
0  Era 1: 2000–2004   1300
1  Era 2: 2005–2009   1304
2  Era 3: 2010–2014   1304
3  Era 4: 2015–2019   1304
4      Era 5: 2020–   1540

Sample of observations per month:
   year_month  count
0     2000-01     21
1     2000-02     21
2     2000-03     23
3     2000-04     20
4     2000-05     23
..        ...    ...
19    2001-08     23
20    2001-09     20
21    2001-10     23
22    2001-11     22
23    2001-12     21

[24 rows x 2 columns]


In [ ]:
# 10. Range counting helper
def count_in_range(start_date: str, end_date: str) -> int:
    mask = (df["date"] >= pd.to_datetime(start_date)) & (df["date"] <= pd.to_datetime(end_date))
    return int(mask.sum())

candidate_ranges = [
    ("candidate_train_1", "2000-01-03", "2015-12-31"),
    ("candidate_val_1", "2016-01-01", "2019-12-31"),
    ("candidate_test_1", "2020-01-01", date_max.strftime("%Y-%m-%d")),
]

rows = []
for name, start, end in candidate_ranges:
    rows.append({"name": name, "start": start, "end": end, "count": count_in_range(start, end)})

candidate_ranges_df = pd.DataFrame(rows)
display(candidate_ranges_df)


=== CANDIDATE RANGE COUNTS (for later split design) ===
                name       start         end  count
0  candidate_train_1  2000-01-03  2015-12-31   4169
1    candidate_val_1  2016-01-01  2019-12-31   1043
2   candidate_test_1  2020-01-01  2025-11-25   1540


In [ ]:
# 11. Walk-forward fold examples (for inspection)
example_folds = [
    ("F1_example", "2000-01-03", "2009-12-31", "2010-01-01", "2011-12-31"),
    ("F2_example", "2000-01-03", "2011-12-31", "2012-01-01", "2013-12-31"),
    ("F3_example", "2000-01-03", "2013-12-31", "2014-01-01", "2015-12-31"),
    ("F4_example", "2000-01-03", "2015-12-31", "2016-01-01", "2019-12-31"),
]

fold_rows = []
for name, tr_start, tr_end, va_start, va_end in example_folds:
    train_mask = (df["date"] >= pd.to_datetime(tr_start)) & (df["date"] <= pd.to_datetime(tr_end))
    val_mask = (df["date"] >= pd.to_datetime(va_start)) & (df["date"] <= pd.to_datetime(va_end))
    fold_rows.append({
        "fold": name,
        "train_start": tr_start,
        "train_end": tr_end,
        "train_count": int(train_mask.sum()),
        "val_start": va_start,
        "val_end": va_end,
        "val_count": int(val_mask.sum()),
        "overlap_train_val": bool(((df["date"] >= pd.to_datetime(va_start)) & (df["date"] <= pd.to_datetime(tr_end))).any()),
    })

fold_counts_df = pd.DataFrame(fold_rows)
display(fold_counts_df)

# Visualize fold sizes
fig_folds = px.bar(
    fold_counts_df.melt(id_vars=["fold"], value_vars=["train_count", "val_count"], var_name="set", value_name="count"),
    x="fold", y="count", color="set", barmode="group", title="Example walk-forward fold sizes"
)
fig_folds.show()


=== GENERIC WALK-FORWARD FOLD COUNTS (EXAMPLE) ===
         fold train_start   train_end  train_count   val_start     val_end  val_count  overlap_train_val
0  F1_example  2000-01-03  2009-12-31         2604  2010-01-01  2011-12-31        521              False
1  F2_example  2000-01-03  2011-12-31         3125  2012-01-01  2013-12-31        522              False
2  F3_example  2000-01-03  2013-12-31         3647  2014-01-01  2015-12-31        522              False
3  F4_example  2000-01-03  2015-12-31         4169  2016-01-01  2019-12-31       1043              False


In [ ]:
# 12. Final quick sanity checks
critical_cols = ["open", "high", "low", "close", "log_return", "simple_return", "vol_30", "vol_90"]
nan_summary = df[critical_cols].isna().sum()
print("NaN counts in critical columns (note: rolling vols and returns naturally have NaNs at start):")
display(nan_summary)




=== FINAL QUICK SANITY CHECKS ===
NaN counts in critical columns (log_return, vol windows will naturally have NaNs at the start):
open              0
high              0
low               0
close             0
log_return        1
simple_return     1
vol_30           30
vol_90           90
dtype: int64

Week 4 EDA (Plotly version) notebook finished. Use the printed tables and plots to decide:
- Final modeling period (start/end).
- Exact train/val/test dates for the simple chronological split.
- Exact fold definitions for the walk-forward validation scheme.
